In [ ]:
# Install (only if needed)
# !pip install nltk scikit-learn torch

import nltk
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

In [ ]:
nltk.download('movie_reviews')

from nltk.corpus import movie_reviews

texts = []
labels = []

for category in movie_reviews.categories():
    for fileid in movie_reviews.fileids(category):
        texts.append(movie_reviews.raw(fileid))
        labels.append(1 if category == 'pos' else 0)

[nltk_data] Downloading package movie_reviews to /root/nltk_data...
[nltk_data]   Unzipping corpora/movie_reviews.zip.


In [ ]:
vectorizer = CountVectorizer(max_features=3000)
X = vectorizer.fit_transform(texts).toarray()
y = labels

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
class TextDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [ ]:
train_loader = DataLoader(TextDataset(X_train, y_train), batch_size=32, shuffle=True)
test_loader = DataLoader(TextDataset(X_test, y_test), batch_size=32)

In [ ]:
class CNNModel(nn.Module):
    def __init__(self, input_size):
        super().__init__()

        kernels = 8
        kernel_size = 3
        self.conv = nn.Conv1d(1, kernels, kernel_size)

        self.pool = nn.MaxPool1d(2)

        # Output size calculation
        conv_output = input_size - kernel_size + 1
        pooled_output = conv_output // 2
        flattened_size = kernels * pooled_output

        self.fc1 = nn.Linear(flattened_size, 16)
        self.fc2 = nn.Linear(16, 1)

        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = x.unsqueeze(1)
        x = torch.relu(self.conv(x))
        x = self.pool(x)

        x = x.view(x.size(0), -1)

        x = torch.relu(self.fc1(x))
        x = self.fc2(x)

        return self.sigmoid(x)

In [ ]:
model = CNNModel(input_size=X.shape[1])

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
for epoch in range(5):
    model.train()

    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()

        output = model(X_batch).squeeze()
        loss = criterion(output, y_batch)

        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

Epoch 1, Loss: 0.4577
Epoch 2, Loss: 0.2494
Epoch 3, Loss: 0.2387
Epoch 4, Loss: 0.3186
Epoch 5, Loss: 0.1026


In [ ]:
y_pred = []
y_actual = []

model.eval()
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        output = model(X_batch).squeeze()
        preds = (output > 0.5).int()

        y_pred.extend(preds.tolist())
        y_actual.extend(y_batch.tolist())

y_actual = [int(y) for y in y_actual]

In [ ]:
print("\nConfusion Matrix:")
print(confusion_matrix(y_actual, y_pred))

print("\nClassification Report:")
print(classification_report(y_actual, y_pred))

print("\nAccuracy Score:", accuracy_score(y_actual, y_pred))


Confusion Matrix:
[[134  65]
 [ 23 178]]

Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.67      0.75       199
           1       0.73      0.89      0.80       201

    accuracy                           0.78       400
   macro avg       0.79      0.78      0.78       400
weighted avg       0.79      0.78      0.78       400


Accuracy Score: 0.78
